# Projeto de Inteligencia Artificial: Predicao de Sobrevivencia no Titanic

Este notebook consolida o projeto no formato solicitado no escopo da disciplina de Inteligencia Artificial.

**Problema:** prever se um passageiro sobreviveria ao naufragio do Titanic com base em dados historicos.

**Tipo de aprendizagem:** aprendizagem supervisionada, pois o conjunto de treino possui a variavel alvo `Survived`.

**Tarefa:** classificacao binaria (`0 = nao sobreviveu`, `1 = sobreviveu`).

> Observacao importante: o escopo cita a analise de 2 conjuntos de dados. Neste projeto, foram usados os dois arquivos oficiais do problema Titanic: `train.csv` para treino/validacao e `test.csv` para predicao final. Caso o professor exija dois datasets independentes, sera necessario adicionar uma segunda base com outro problema.

## 1. Fundamentacao teorica

A Aprendizagem de Maquina permite que algoritmos identifiquem padroes em dados historicos e usem esses padroes para prever novos casos. No contexto do Titanic, cada registro representa um passageiro e suas caracteristicas, como classe da passagem, sexo, idade, familiares a bordo, tarifa paga e porto de embarque.

Como a variavel `Survived` ja e conhecida no conjunto de treino, o problema e de **aprendizagem supervisionada**. Como essa variavel possui apenas duas classes, trata-se de uma **classificacao binaria**.

Os modelos comparados neste notebook sao:

- Regressao Logistica
- KNN
- Random Forest
- XGBoost
- LightGBM

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings('ignore')

if not os.path.exists('datas') and os.path.exists('../datas'):
    os.chdir('..')

train_df = pd.read_csv('datas/train.csv')
test_df = pd.read_csv('datas/test.csv')
gender_submission_df = pd.read_csv('datas/gender_submission.csv')

print('Conjunto de treino:', train_df.shape)
print('Conjunto de teste:', test_df.shape)
print('Arquivo de exemplo de submissao:', gender_submission_df.shape)

## 2. Metodologia: escolha e analise dos conjuntos de dados

A base utilizada foi o dataset Titanic, disponibilizado pelo Kaggle. O conjunto de treino contem passageiros com a classe real de sobrevivencia. O conjunto de teste contem passageiros sem o rotulo, usado para gerar as predicoes finais.

Antes do treinamento, e importante verificar tamanho dos dados, tipos de variaveis, valores ausentes e balanceamento das classes.

In [ ]:
display(train_df.head())
display(train_df.info())
display(train_df.describe(include='all'))

In [ ]:
class_balance = train_df['Survived'].value_counts().rename(index={0: 'Nao sobreviveu', 1: 'Sobreviveu'})
class_balance_percent = (train_df['Survived'].value_counts(normalize=True) * 100).rename(index={0: 'Nao sobreviveu (%)', 1: 'Sobreviveu (%)'})

balance_df = pd.concat([class_balance, class_balance_percent.round(2)], axis=0).to_frame('valor')
display(balance_df)

missing_df = pd.DataFrame({
    'ausentes_treino': train_df.isna().sum(),
    'ausentes_teste': test_df.isna().sum()
}).fillna(0).astype(int)
display(missing_df[missing_df.sum(axis=1) > 0])

## 3. Pre-processamento

Foram selecionadas variaveis com boa disponibilidade e utilidade preditiva:

- `Pclass`: classe da passagem
- `Sex`: sexo do passageiro
- `Age`: idade
- `SibSp`: irmaos/conjuges a bordo
- `Parch`: pais/filhos a bordo
- `Fare`: tarifa paga
- `Embarked`: porto de embarque

Tratamentos aplicados:

- preenchimento de idade com a mediana do treino;
- preenchimento de tarifa com a mediana do treino;
- preenchimento de porto de embarque com a moda do treino;
- codificacao de `Sex` e `Embarked` para valores numericos;
- separacao treino/validacao com estratificacao da classe alvo.

In [ ]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
target = 'Survived'

age_median = train_df['Age'].median()
fare_median = train_df['Fare'].median()
embarked_mode = train_df['Embarked'].mode()[0]

def preprocess(df):
    data = df[features].copy()
    data['Age'] = data['Age'].fillna(age_median)
    data['Fare'] = data['Fare'].fillna(fare_median)
    data['Embarked'] = data['Embarked'].fillna(embarked_mode)
    data['Sex'] = data['Sex'].map({'male': 0, 'female': 1})
    data = pd.get_dummies(data, columns=['Embarked'], drop_first=False)
    return data

X = preprocess(train_df)
y = train_df[target]
X_test_final = preprocess(test_df)
X_test_final = X_test_final.reindex(columns=X.columns, fill_value=0)

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Treino:', X_train.shape)
print('Validacao:', X_valid.shape)
print('Teste final:', X_test_final.shape)
display(X_train.head())

## 4. Tabela de hiperparametros

A tabela abaixo registra os principais hiperparametros usados em cada modelo, conforme solicitado no escopo.

In [ ]:
hyperparams_df = pd.DataFrame([
    {'Modelo': 'Regressao Logistica', 'Parametro': 'max_iter', 'Valor': 1000},
    {'Modelo': 'Regressao Logistica', 'Parametro': 'random_state', 'Valor': 42},
    {'Modelo': 'KNN', 'Parametro': 'n_neighbors', 'Valor': 7},
    {'Modelo': 'KNN', 'Parametro': 'scaler', 'Valor': 'StandardScaler'},
    {'Modelo': 'Random Forest', 'Parametro': 'n_estimators', 'Valor': 300},
    {'Modelo': 'Random Forest', 'Parametro': 'max_depth', 'Valor': 6},
    {'Modelo': 'Random Forest', 'Parametro': 'min_samples_split', 'Valor': 5},
    {'Modelo': 'Random Forest', 'Parametro': 'min_samples_leaf', 'Valor': 2},
    {'Modelo': 'XGBoost', 'Parametro': 'n_estimators', 'Valor': 300},
    {'Modelo': 'XGBoost', 'Parametro': 'max_depth', 'Valor': 4},
    {'Modelo': 'XGBoost', 'Parametro': 'learning_rate', 'Valor': 0.05},
    {'Modelo': 'LightGBM', 'Parametro': 'n_estimators', 'Valor': 300},
    {'Modelo': 'LightGBM', 'Parametro': 'learning_rate', 'Valor': 0.05},
    {'Modelo': 'LightGBM', 'Parametro': 'max_depth', 'Valor': 4},
    {'Modelo': 'LightGBM', 'Parametro': 'num_leaves', 'Valor': 15},
])

display(hyperparams_df)

## 5. Treinamento e avaliacao dos modelos

Foram usadas quatro metricas de classificacao:

- **Acuracia:** proporcao total de acertos;
- **Precisao:** entre os previstos como sobreviventes, quantos realmente sobreviveram;
- **Recall:** entre os sobreviventes reais, quantos o modelo encontrou;
- **F1-score:** media harmonica entre precisao e recall.

A matriz de confusao tambem e apresentada para cada modelo, pois o problema e de classificacao.

In [ ]:
models = {
    'Regressao Logistica': LogisticRegression(max_iter=1000, random_state=42),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=7))
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42
    )
}

try:
    from xgboost import XGBClassifier
    models['XGBoost'] = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric='logloss',
        random_state=42
    )
except Exception as exc:
    print('XGBoost nao disponivel:', exc)

try:
    from lightgbm import LGBMClassifier
    models['LightGBM'] = LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        num_leaves=15,
        random_state=42,
        verbosity=-1
    )
except Exception as exc:
    print('LightGBM nao disponivel:', exc)

results = []
confusion_matrices = {}
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_valid)
    fitted_models[name] = model
    confusion_matrices[name] = confusion_matrix(y_valid, pred)
    results.append({
        'Modelo': name,
        'Acuracia': accuracy_score(y_valid, pred),
        'Precisao': precision_score(y_valid, pred, zero_division=0),
        'Recall': recall_score(y_valid, pred, zero_division=0),
        'F1-score': f1_score(y_valid, pred, zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values('F1-score', ascending=False).reset_index(drop=True)
display(results_df.style.format({
    'Acuracia': '{:.4f}',
    'Precisao': '{:.4f}',
    'Recall': '{:.4f}',
    'F1-score': '{:.4f}'
}))

In [ ]:
for model_name, cm in confusion_matrices.items():
    print(f'\nMatriz de confusao - {model_name}')
    cm_df = pd.DataFrame(cm, index=['Real 0', 'Real 1'], columns=['Predito 0', 'Predito 1'])
    display(cm_df)

In [ ]:
best_model_name = results_df.iloc[0]['Modelo']
best_model = fitted_models[best_model_name]

print('Melhor modelo pela metrica F1-score:', best_model_name)
print('\nRelatorio de classificacao do melhor modelo:')
print(classification_report(y_valid, best_model.predict(X_valid), zero_division=0))

## 6. Geracao da predicao final

Apos selecionar o melhor modelo na validacao, ele e usado para gerar as predicoes do conjunto de teste oficial.

In [ ]:
final_predictions = best_model.predict(X_test_final)
submission_df = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': final_predictions
})

output_path = 'submission_projeto_ia_titanic.csv'
submission_df.to_csv(output_path, index=False)

print('Arquivo gerado:', output_path)
display(submission_df.head())

## 7. Conclusao

O projeto aplicou diferentes algoritmos de Machine Learning a um problema de classificacao binaria. A comparacao entre modelos foi feita com acuracia, precisao, recall e F1-score, alem das matrizes de confusao.

O modelo mais adequado deve ser escolhido com base no desempenho da tabela comparativa. Para este problema, o F1-score e uma metrica importante porque considera o equilibrio entre precisao e recall na classe de sobreviventes.

Como melhoria futura, seria possivel testar mais engenharia de atributos, como extrair titulos dos nomes (`Mr`, `Mrs`, `Miss`), tamanho da familia, informacoes da cabine e validacao cruzada.